# Исследование надежности заемщиков


Во второй части проекта вы выполните шаги 3 и 4. 
Чтобы вам не пришлось писать код заново для шагов 1 и 2, мы добавили авторские решения в ячейки с кодом. 



## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`../data/data.csv`

In [ ]:
import pandas as pd

try:
    data = pd.read_csv('../data/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [ ]:
data.head(20)

    children  days_employed  dob_years            education  education_id  \
0          1   -8437.673028         42               высшее             0   
1          1   -4024.803754         36              среднее             1   
2          0   -5623.422610         33              Среднее             1   
3          3   -4124.747207         32              среднее             1   
4          0  340266.072047         53              среднее             1   
5          0    -926.185831         27               высшее             0   
6          0   -2879.202052         43               высшее             0   
7          0    -152.779569         50              СРЕДНЕЕ             1   
8          2   -6929.865299         35               ВЫСШЕЕ             0   
9          0   -2188.756445         41              среднее             1   
10         2   -4171.483647         36               высшее             0   
11         0    -792.701887         40              среднее             1   

**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [ ]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [ ]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [ ]:
data['days_employed'] = data['days_employed'].abs()

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [ ]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [ ]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [ ]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [ ]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.**

In [ ]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [ ]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [ ]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру.**

In [ ]:
data['education'] = data['education'].str.lower()

**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [ ]:
data.duplicated().sum()

71

In [ ]:
data = data.drop_duplicates()

### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [ ]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [ ]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [ ]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [ ]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [ ]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)
data.head(20)

    children  days_employed  dob_years            education  education_id  \
0          1    8437.673028         42               высшее             0   
1          1    4024.803754         36              среднее             1   
2          0    5623.422610         33              среднее             1   
3          3    4124.747207         32              среднее             1   
4          0  340266.072047         53              среднее             1   
5          0     926.185831         27               высшее             0   
6          0    2879.202052         43               высшее             0   
7          0     152.779569         50              среднее             1   
8          2    6929.865299         35               высшее             0   
9          0    2188.756445         41              среднее             1   
10         2    4171.483647         36               высшее             0   
11         0     792.701887         40              среднее             1   

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

In [ ]:
#Для начала напишем функцию, которая возвращает таблицу с данными для анализа. 
#Так как задания однотипные эту функцию можно будет применить к следующим заданиям.
def rating(row, param):# Принимает два аргумента для функции: row и param
    data = row.groupby(param).agg({'debt':'sum'})# Группирует данные в row по параметру param с помощью метода groupby и 
    # агрегатную функцию sum к столбцу debt, чтобы получить сумму долгов для каждой группы.
    data['count'] = row.groupby(param)['debt'].count()#Создаёт новый столбец count, который содержит количество элементов в 
    # каждой группе. Для этого используется метод groupby и атрибут count.
    data['percent'] = data['debt'] / data['count'] * 100 # Вычисляет процент долга для каждой группы, разделив сумму долга на количество элементов и умножив на 100.
    return data
data

       children  days_employed  dob_years education  education_id  \
0             1    8437.673028         42    высшее             0   
1             1    4024.803754         36   среднее             1   
2             0    5623.422610         33   среднее             1   
3             3    4124.747207         32   среднее             1   
4             0  340266.072047         53   среднее             1   
...         ...            ...        ...       ...           ...   
21520         1    4529.316663         43   среднее             1   
21521         0  343937.404131         67   среднее             1   
21522         1    2113.346888         38   среднее             1   
21523         3    3112.481705         38   среднее             1   
21524         2    1984.507589         40   среднее             1   

          family_status  family_status_id gender income_type  debt  \
0       женат / замужем                 0      F   сотрудник     0   
1       женат / замужем        

In [ ]:
data_children = rating(data,'children')
data_children.sort_values(by='percent',ascending=True)

          debt  count   percent
children                       
5            0      9  0.000000
0         1063  14091  7.543822
3           27    330  8.181818
1          444   4808  9.234609
2          194   2052  9.454191
4            4     41  9.756098

**Вывод: Заемщики, не имеющие детей (70-75% от общего числа заемщиков) выплачивают кредит эффективнее, чем те, у кого имеются дети. Скорее всего это связанно с незапланированными тратами у заемщиков, у которых есть дети** 

In [ ]:
#в таком случае, малочисленные данные можно убрать из сводной таблицы, аналогично таблице data_total_income.
data_children.loc[data_children['count'] >= 100].sort_values(by = 'percent', ascending = True)

          debt  count   percent
children                       
0         1063  14091  7.543822
3           27    330  8.181818
1          444   4808  9.234609
2          194   2052  9.454191

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [ ]:
data_family_status = rating(data,'family_status')
data_family_status.sort_values(by = 'percent', ascending = True)# Ваш код будет здесь. Вы можете создавать новые ячейки.

                       debt  count   percent
family_status                               
вдовец / вдова           63    951  6.624606
в разводе                84   1189  7.064760
женат / замужем         927  12261  7.560558
гражданский брак        385   4134  9.313014
Не женат / не замужем   273   2796  9.763948

**Вывод: Заемщики с наименьшими задолжностями по платежам- это люди категории "вдовец / вдова", в то время как заемщики не находяшиеся в отношениях имеют задолженности в 1.45 раз выше. Скорее всего это связанно с тем, что люди находяшиеся/находившиеся в отношения - более внимательны к своим тратамб более ответственные и дисциплинированые. При этом заемщики категории "гражданский брак" имеют показатели приближенные к заемщикахб которые не находяятся в отношениях, возможно это связанно с тем, что на паре не лежит груз ответсвенностибб так как не защищен законом, как обычный брак, из-за ограниченных обязательств перед супругом/супругой.** 

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [ ]:
data_total_income = rating(data, 'total_income_category')# Ваш код будет здесь. Вы можете создавать новые ячейки.
data_total_income.sort_values(by = 'percent', ascending = True)

                       debt  count   percent
total_income_category                       
D                        21    349  6.017192
B                       354   5014  7.060231
A                         2     25  8.000000
C                      1353  15921  8.498210
E                         2     22  9.090909

In [ ]:
#в таблице имеются категории которые значительно отличаются количеством от остальных, их меньше ста, для удобного анализа уберем их.
data_total_income.loc[data_total_income['count'] >= 100].sort_values(by = 'percent', ascending = True)

                       debt  count   percent
total_income_category                       
D                        21    349  6.017192
B                       354   5014  7.060231
C                      1353  15921  8.498210

**Вывод: Исходя из полученной таблицы можем сделать вывод, что люди с доходом 30001-50000 рублей выплачивают кредит в сроки, при этом заемщики с доходом 200001-1000000 рублей составляют около 75% от общего числа.** 

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

In [ ]:
data_total_purpose = rating(data, 'purpose_category')# Ваш код будет здесь. Вы можете создавать новые ячейки.
data_total_purpose.sort_values(by = 'percent')

                          debt  count   percent
purpose_category                               
операции с недвижимостью   780  10751  7.255139
проведение свадьбы         183   2313  7.911803
получение образования      369   3988  9.252758
операции с автомобилем     400   4279  9.347978

**Вывод: Операции связанные с недвижимостью имеют самый низкий процент задолженности, а также самым количественным по числу заемщиков.** 

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

*Ответ: Пропуски встречались в колонках days_employed и total_income в обоих по 2174 пропуска, это может быть связаны с человеческим фактором, например оператор не внес данные о рабочем стаже и доходе заемщика, но это врядли, та как данные необходиимы для обобрения кредита. Скорее всего пропуски связаны со сбоем при выгрузке данных, либо сбой приложения отвечающего за автоматическую выгрузку базы данных, то есть прчина технологическая.* 

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ: Пропуски в базах даных можно:
1)Не трогать. В таком случае, будут искажаться данные, например при подсчете общего дохода мы ошибемся, посчитав пустые яцейки
2)Удалить строки. Но в таком случае мы потеряем важные данные, которые необходимы для анализа.
3)Заменить данные на среднее арифметическое или медианное значение. Медианное значение подходит лучше, так как в отличе от среднего арифметического значения, медианное не так чувствительно к критически высоким показателям и на нее не влияют выбросы(случайными значениями я подразумевал именно их)* 

### Шаг 4: общий вывод.

В рамках проекта для улучшения системы скоринга клиентов были: 
1)предобработан датафрейм с информацией о клиентах; 
2)были созданы дополнительные колонки "категория цели креита" и "категория общего дохода" для удобного и ясного анализа данных; 3)проанализированы зависимости столбцов датафрейма. 
Исходя из проекта можно сделать вывод, что кредит лучше выдавать клиенту, который женат/замужем, не имеет детей, имеет доход от 30001 до 50000 рублейб и который собирается приобрести недвижимость. А вот к клиентам неженатым/незамужним, с доходом 200001-1000000 рублей, занимающим деньги на операции с автомобилем- кредит выдавать на свой страх и риск.